# Analsis exploratorio de datos EDA

Se va a examinar, investigar y resumir las caracteristicas principales de la informacion crediticia.


## Objetivo del notebook

Este notebook realiza EDA + EVA para generar artefactos de datos que luego consume el notebook de modelado.

### Entradas
- Base de datos PostgreSQL con tablas creditos, amortizacion y juicios.
- Rango temporal definido en la consulta SQL.

### Salidas
- output/datasets/datos_preprocesados.csv
- output/metricas/recomendaciones_eva.csv
- output/evidencia_eva/reporte_eva.json

### Criterio de exito
- La extraccion no regresa vacio.
- El preprocesamiento genera crisis_flag.
- EVA produce recomendaciones y evidencias.

In [ ]:
## Comandos para autoreload de modulos y librerias
%load_ext autoreload
%autoreload 2

import logging
import os
import polars as pl
import sys
sys.path.insert(0, '..')

from datetime import datetime
from sqlalchemy import create_engine
from src.eva import Pipeline

## Configuracion del notebook

Se configura el notebook para la ejecucion:

- ignorar warnings
- agregar el path del proyecto para importar modulos
- datos de la conexion a la base de datos

In [ ]:
# Configuracion de salida
PATH_SALIDA = "output"

# Configuracion DB
DB_CONFIG = {
    "host": "localhost",
    "port": "5432",
    "database": "postgres_db",
    "user": "postgres_usr",
    "password": "admin123",
}

# Crear directorios para resultados
os.makedirs(f"{PATH_SALIDA}/evidencia_eva", exist_ok=True)
os.makedirs(f"{PATH_SALIDA}/datasets", exist_ok=True)
os.makedirs(f"{PATH_SALIDA}/graficas", exist_ok=True)
os.makedirs(f"{PATH_SALIDA}/metricas", exist_ok=True)
os.makedirs(f"{PATH_SALIDA}/logs", exist_ok=True)

# Configuracion de logging
logger = logging.getLogger(__name__)
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    handlers=[
        logging.FileHandler(f"{PATH_SALIDA}/logs/eda_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"),
        logging.StreamHandler(),
    ],
    force=True,
 )

connection_string = (
    f"postgresql://{DB_CONFIG['user']}:{DB_CONFIG['password']}"
    f"@{DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['database']}"
)
engine = create_engine(connection_string)

print("=" * 80)
print("EDA + EVA PARA GENERACION DE ARTEFACTOS")
print("=" * 80)
print("Salida esperada:")
print(f"- Dataset: ./{PATH_SALIDA}/datasets/datos_preprocesados.csv")
print(f"- Recomendaciones EVA: ./{PATH_SALIDA}/metricas/recomendaciones_eva.csv")
print(f"- Evidencias EVA: ./{PATH_SALIDA}/evidencia_eva/reporte_eva.json")
print("=" * 80)

## Extracción de datos
Se va a extraer la informacion crediticia de la base de datos de la entidad financiera.

In [ ]:
query_completa = """
SELECT
    c.numero_credito,
    c.codigo_act_financiera,
    c.codigo_producto,
    c.fecha_credito,
    c.codigo_perioc,
    c.codigo_orirec,
    c.deb_aut,
    c.cant_soli,
    c.num_cuotas,
    c.tasa_interes,
    c.mora,
    c.tab_amortiza,
    c.tot_dias_mora,       -- INCLUIDO PARA EVA (se analizará su exclusión)
    c.tot_num_moras,       -- INCLUIDO PARA EVA (se analizará su exclusión)
    c.estado_cred as estado_credito,
    c.estado as estado_solicitud,
    c.per_gracia,
    c.capital_porpagar,
    c.mismodia,
    c.numdias,
    c.oficre,
    c.codigo_sucursal,
    c.pigper,
    c.interes_fijo,
    c.porc_pig,
    c.fecini,
    c.fecfin,
    c.judicial,
    c.codigo_grupo,
    c.codigo,
    c.codigo_destino,
    c.codigo_destino_det,
    c.costo_judicial,
    c.notificaciones,
    c.gestion_cobro,
    c.fecha_gestion_cobro,
    c.desem_parc,
    c.monto_real,
    c.saldo_capital,
    a.ordencal,
    a.fecinical,
    a.fecfincal,
    a.saldocal,
    a.capitalcal,
    a.interescal,
    a.diasintcal,
    a.fechaincal,
    a.moracal,
    a.diasmoracal,
    a.fechamoracal,
    a.rubroscal,
    a.totalcal,
    j.codigo_tipo_juicio,
    j.tipo_operacion,
    j.valor_demanda,
    j.capital_recuperado,
    j.fecha_proceso,
    j.fecha_recuperado,
    j.fecha_cierre,
    j.estado as estado_juicio
FROM creditos c
JOIN amortizacion a ON c.numero_credito = a.numero_credito
LEFT JOIN juicios j ON c.numero_credito = j.numero_credito
WHERE c.fecha_credito >= '2020-01-01' 
  AND c.fecha_credito < '2020-02-01'
ORDER BY c.fecha_credito, c.numero_credito, a.ordencal
"""

print("\n📥 Extrayendo datos desde PostgreSQL...")
df_raw = pl.read_database(query=query_completa, connection=engine, infer_schema_length=None)
print(f"✅ Datos extraídos: {len(df_raw):,} registros, {len(df_raw.columns)} columnas")
print(f"📊 Rango de fechas: {df_raw['fecha_credito'].min()} a {df_raw['fecha_credito'].max()}")

### Pre procesamiento y feature engineering

In [ ]:
def preprocesar_datos(df_raw, config=None):
    """
    Preprocesa los datos extraídos para el modelado.
    Incluye agregación por mes-bloque y creación de features.
    """
    logger.info("🔄 Iniciando preprocesamiento de datos...")
    
    df = df_raw.clone()
    
    # 1. Crear columna de mes
    df = df.with_columns(pl.col('fecha_credito').dt.truncate('1mo').alias('mes'))
    
    # 2. Crear bloque_id
    df = df.with_columns(df['codigo_act_financiera'].fill_null('SIN_RIESGO').cast(pl.String).alias('riesgo'))
    df = df.with_columns(df['codigo_producto'].fill_null('SIN_SECTOR').cast(pl.String).alias('sector'))
    df = df.with_columns(
        (df['riesgo'] + '_' + df['sector'] + '_' + df['codigo_sucursal'].cast(pl.String)).alias('bloque_id')
    )
    
    # 2.5 Convertir columnas string a numéricas (NUMERIC de PG viene como String)
    logger.info('🔄 Convirtiendo columnas a tipos numéricos...')
    cols_numericas_str = [
        'cant_soli', 'monto_real', 'tasa_interes', 'mora',
        'capital_porpagar', 'porc_pig', 'costo_judicial',
        'notificaciones', 'saldo_capital', 'saldocal',
        'capitalcal', 'interescal', 'moracal', 'rubroscal',
        'totalcal', 'valor_demanda', 'capital_recuperado',
        'num_cuotas', 'tot_dias_mora', 'tot_num_moras',
        'gestion_cobro'
    ]
    for col_name in cols_numericas_str:
        if col_name in df.columns and df[col_name].dtype == pl.String:
            df = df.with_columns(pl.col(col_name).cast(pl.Float64, strict=False).fill_null(0))
    
    # 3. Agregar por mes-bloque
    logger.info("📊 Agregando datos por mes y bloque...")
    df_agregado = df.group_by(['mes', 'bloque_id', 'riesgo', 'sector', 
                              'codigo_sucursal']).agg([
        # Métricas de crédito
        pl.col('numero_credito').count().alias('num_creditos'),
        pl.col('cant_soli').sum().alias('monto_total'),
        pl.col('monto_real').sum().alias('monto_promedio'),
        pl.col('num_cuotas').mean().alias('plazo_promedio'),
        pl.col('tasa_interes').mean().alias('tasa_interes_promedio'),
        pl.col('saldo_capital').mean().alias('saldo_promedio'),
        pl.col('costo_judicial').sum().alias('total_costo_judicial'),
        pl.col('gestion_cobro').sum().alias('total_gestion_cobro'),
        pl.col('notificaciones').sum().alias('total_notificaciones'),
        # Variables a evaluar en EVA
        pl.col('tot_dias_mora').mean().alias('tot_dias_mora_promedio'),
        pl.col('tot_num_moras').mean().alias('tot_num_moras_promedio'),
        pl.col('mora').mean().alias('mora_promedio'),
        # Indicadores
        (pl.col('judicial') == 'S').sum().alias('creditos_judiciales'),
        pl.col('estado_credito').is_in(['C', 'L']).sum().alias('creditos_cerrados'),
    ])
    
    
    
    # ===== NUEVO: CONVERTIR A NUMÉRICO ANTES DE CALCULAR =====
    logger.info("🔄 Convirtiendo columnas a tipos numéricos...")
    cols_a_convertir = [
        'num_creditos', 'monto_total', 'monto_promedio', 'plazo_promedio',
        'tasa_interes_promedio', 'saldo_promedio', 'total_costo_judicial',
        'total_gestion_cobro', 'total_notificaciones', 'tot_dias_mora_promedio',
        'tot_num_moras_promedio', 'mora_promedio', 'creditos_judiciales',
        'creditos_cerrados'
    ]
    
    df_agregado = df_agregado.with_columns([
        pl.col(col).cast(pl.Float64, strict=False).fill_null(0).alias(col)
        for col in cols_a_convertir
    ])
    
    # 4. Calcular tasas y ratios (como float)
    logger.info("🔄 Calculando tasas y ratios...")
    df_agregado = df_agregado.with_columns(
        pl.when(pl.col('num_creditos') > 0)
        .then((pl.col('creditos_judiciales') / pl.col('num_creditos')) * 100)
        .otherwise(0.0).cast(pl.Float64).alias('tasa_judicial')
    )
    
    df_agregado = df_agregado.with_columns(
        pl.when(pl.col('num_creditos') > 0)
        .then((pl.col('creditos_cerrados') / pl.col('num_creditos')) * 100)
        .otherwise(0.0).cast(pl.Float64).alias('tasa_cierre')
    )
    
    # 5. Calcular crisis_flag
    logger.info("🎯 Calculando crisis_flag...")
    df_agregado = df_agregado.with_columns(
        (
            (pl.col('tasa_judicial') > 5).cast(pl.Int64) * 3 +
            pl.when(pl.col('tasa_judicial') > 2).then(1).otherwise(0) +
            pl.when(pl.col('total_costo_judicial') > 0)
              .then(pl.when(pl.col('total_costo_judicial') / pl.col('num_creditos') > pl.col('monto_promedio') * 0.1)
                    .then(2).otherwise(0))
              .otherwise(0) +
            pl.when(pl.col('total_gestion_cobro') > 0)
              .then(pl.when(pl.col('total_gestion_cobro') / pl.col('num_creditos') > pl.col('monto_promedio') * 0.05)
                    .then(1).otherwise(0))
              .otherwise(0) +
            pl.when(pl.col('tasa_cierre') > 30).then(2)
              .when(pl.col('tasa_cierre') > 20).then(1)
              .otherwise(0) +
            pl.when(pl.col('plazo_promedio') > 36).then(1).otherwise(0) +
            pl.when(pl.col('plazo_promedio') > 60).then(1).otherwise(0) +
            pl.when(pl.col('tasa_interes_promedio') > 15).then(1).otherwise(0)
        ).ge(5).cast(pl.Int64).alias('crisis_flag')
    )
    
    # 6. Ordenar por bloque y mes
    df_agregado = df_agregado.sort(['bloque_id', 'mes'])
    
    logger.info(f"✅ Preprocesamiento completado: {len(df_agregado):,} registros, {len(df_agregado.columns)} columnas")
    logger.info(f"📊 Tipos de datos:\n{df_agregado.dtypes}")
    logger.info(f"📊 Distribución crisis_flag: {df_agregado['crisis_flag'].value_counts().to_pandas().set_index('crisis_flag')['count'].to_dict()}")
    
    return df_agregado

## Ejecucion principal EDA + EVA

Funcion principal que ejecuta extraccion, preprocesamiento, analisis EVA y exportacion de artefactos para el notebook de modelado.

In [ ]:
def main_eva(modo='notebook'):
    """
    Pipeline completo: extraccion -> preprocesamiento -> EVA -> exportacion de artefactos.

    Parameters
    ----------
    modo : str
        'notebook' -> salida a consola + disco (.png, .csv)
        'mlflow'   -> envia metricas, figura y artefactos a MLflow
    """
    import json

    logger.info("INICIANDO PIPELINE COMPLETO - EDA + EVA")
    logger.info("-" * 10)

    # 1. Extraer datos
    logger.info("1) EXTRAYENDO DATOS DESDE POSTGRESQL")
    df_raw = pl.read_database(query=query_completa, connection=engine, infer_schema_length=None)
    logger.info(f"Datos extraidos: {len(df_raw):,} registros, {len(df_raw.columns)} columnas")

    if df_raw.is_empty():
        raise ValueError(
            "La consulta no devolvio registros. Revisa fechas, tablas origen o conexion a PostgreSQL."
        )

    # 2. Preprocesar
    logger.info("2) PREPROCESANDO DATOS")
    df_features = preprocesar_datos(df_raw)

    # 3. Ejecutar EVA
    logger.info("3) EJECUTANDO EVA (modo: %s)", modo)
    pipeline = Pipeline(
        modo=modo,
        output_dir=f"{PATH_SALIDA}",
    )
    recomendaciones, evidencias = pipeline.run(df_features, target_col='crisis_flag')

    # 4. Guardar artefactos para modelado
    logger.info("4) GUARDANDO ARTEFACTOS PARA MODELADO")
    dataset_path = f"{PATH_SALIDA}/datasets/datos_preprocesados.csv"
    evidencias_path = f"{PATH_SALIDA}/evidencia_eva/reporte_eva.json"

    df_features.write_csv(dataset_path)

    with open(evidencias_path, "w", encoding="utf-8") as f:
        json.dump(
            {
                "recomendaciones": recomendaciones,
                "evidencias": evidencias,
                "filas_dataset": len(df_features),
                "columnas_dataset": len(df_features.columns),
                "fecha_generacion": datetime.now().isoformat(),
            },
            f,
            indent=2,
            ensure_ascii=False,
            default=str,
        )

    logger.info("Artefactos generados:")
    logger.info("- %s", dataset_path)
    logger.info("- %s", f"{PATH_SALIDA}/metricas/recomendaciones_eva.csv")
    logger.info("- %s", evidencias_path)

    return df_features, recomendaciones

### Ejecutar

In [ ]:
print("ANALISIS EDA + EVA")
print("-" * 50)
print("Autor: omar.velez@yachaytech.edu.ec")
print("Fecha: 2026")
print("-" * 50)

# Ejecutar pipeline EDA + EVA
df_features, recomendaciones = main_eva()

print("\nPIPELINE EDA + EVA COMPLETADO")
print("Artefactos generados:")
print("1) output/datasets/datos_preprocesados.csv")
print("2) output/metricas/recomendaciones_eva.csv")
print("3) output/evidencia_eva/reporte_eva.json")
print("-" * 50)

## Resumen de ejecucion (llenar en cada corrida)

- Fecha de ejecucion:
- Registros de entrada (df_raw):
- Registros de salida (df_features):
- Distribucion de crisis_flag:
- Alertas relevantes encontradas:
- Archivos generados:
  - output/datasets/datos_preprocesados.csv
  - output/metricas/recomendaciones_eva.csv
  - output/evidencia_eva/reporte_eva.json
- Proximo paso: ejecutar notebook de modelado usando el dataset generado.

![icon](../../DocumentosBase/yachayCuadrado.jpg)<br/>***<omar.velez@yachaytech.edu.ec>***<br/>*julio 2026*